In [3]:
import pandas as pd
import re

df_c = pd.read_csv('customs_hs72.csv', encoding='utf-8-sig', low_memory=False)
print(f"원본: {len(df)}행")

원본: 35855행


In [4]:
df_c.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35855 entries, 0 to 35854
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   year           35855 non-null  int64  
 1   month          35855 non-null  int64  
 2   day            35855 non-null  int64  
 3   country        35855 non-null  object 
 4   hs_code        35855 non-null  int64  
 5   export_value   35855 non-null  int64  
 6   export_weight  35855 non-null  float64
 7   import_value   35855 non-null  int64  
 8   import_weight  35855 non-null  float64
 9   unit_price     26738 non-null  float64
 10  date           35855 non-null  object 
 11  price_chg_3m   24439 non-null  float64
 12  vol_chg_3m     24447 non-null  float64
 13  price_chg_6m   23976 non-null  float64
 14  vol_chg_6m     23980 non-null  float64
 15  price_chg_12m  23083 non-null  float64
 16  vol_chg_12m    23086 non-null  float64
dtypes: float64(9), int64(6), object(2)
memory usage: 4

### 1. 국가명 통일

In [5]:
country_map = {
    'USA':      'US',
    'Canada':   'CA',
    'UK':       'GB',
    'Thailand': 'TH',
    'China':    'CN',
    'India':    'IN',
    'EU':       'EU'
}
df_c['country'] = df_c['country'].replace(country_map)
print(f"국가 분포:\n{df_c['country'].value_counts()}")

국가 분포:
country
EU    5488
US    5485
CN    5290
IN    5289
TH    4898
GB    4703
CA    4702
Name: count, dtype: int64


### 2. 날짜 정리

In [6]:
df_c['year_month'] = pd.to_datetime(
    df_c['date'], format='%Y.%m.%d'
).dt.to_period('M')
df_c = df_c.drop(columns=['year', 'month', 'day', 'date'])

### 3. HS코드 문자열 통일

In [8]:
df_c['hs_code'] = df_c['hs_code'].astype(str)

### 4. unit_price 결측 확인

In [11]:
print(f"\nunit_price 결측: {df_c['unit_price'].isna().sum()}건 ({df_c['unit_price'].isna().mean()*100:.1f}%)")
zero_export = df_c[df_c['export_value'] == 0]
print(f"수출액 0인 행: {len(zero_export)}건")
print(f"이 중 unit_price 결측: {zero_export['unit_price'].isna().sum()}건")


unit_price 결측: 9117건 (25.4%)
수출액 0인 행: 9115건
이 중 unit_price 결측: 9115건


In [12]:
df_c = df_c[df_c['export_value'] > 0]
print(f"제거 후: {len(df_c)}행")

제거 후: 26740행


### 5. 변화율 컬럼 결측 확인

In [14]:
chg_cols = ['price_chg_3m','vol_chg_3m','price_chg_6m',
            'vol_chg_6m','price_chg_12m','vol_chg_12m']
print(f"\n변화율 컬럼 결측값:")
print(df_c[chg_cols].isnull().sum())


변화율 컬럼 결측값:
price_chg_3m     2301
vol_chg_3m       2298
price_chg_6m     2764
vol_chg_6m       2761
price_chg_12m    3657
vol_chg_12m      3657
dtype: int64


### 6. Import 컬럼 제거

In [15]:
df_c = df_c.drop(columns=['import_value', 'import_weight'])

### 7. 저장

In [18]:
df_c.to_csv('customs_cleaned.csv', index=False, encoding='utf-8-sig')
print("\n✅ customs_cleaned.csv 저장 완료")


✅ customs_cleaned.csv 저장 완료
